# 加载数据

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

train_files = [
    "test_12_ssvep",
    "test_13_ssvep",
    "test_14_ssvep",
    "test_15_ssvep",
    # "test_6_ssvep",
    # "test_7_ssvep",
]
test_files = ['test_11_ssvep']

freqs = [15, 12, 10]


def load_files(file_list):
    Xs, ys = [], []

    for txt in file_list:
        x = np.load(f'data/eeg_trials_{txt}.npy', allow_pickle=True)
        y = np.load(f'data/eeg_labels_{txt}.npy')

        x = np.asarray(x)
        if x.dtype == object:
            x = np.stack(x)

        y = y.astype(int)
        # y = (y != 0).astype(np.int64)

        print(txt, x.shape, y.shape, np.bincount(y))

        Xs.append(x)
        ys.append(y)

    return Xs, ys


train_Xs, train_ys = load_files(train_files)
test_Xs, test_ys = load_files(test_files)

# 所有文件统一时间长度
min_T = min([x.shape[2] for x in train_Xs + test_Xs])

train_Xs = [x[:, :, :min_T] for x in train_Xs]
test_Xs = [x[:, :, :min_T] for x in test_Xs]

X_train_all = np.concatenate(train_Xs, axis=0).astype(np.float32)
y_train_all = np.concatenate(train_ys, axis=0).astype(np.int64)

X_test = np.concatenate(test_Xs, axis=0).astype(np.float32)
y_test = np.concatenate(test_ys, axis=0).astype(np.int64)

print("X_train_all:", X_train_all.shape)
print("y_train_all:", y_train_all.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

# 前3个文件再切 train / val
X_train, X_val, y_train, y_val = train_test_split(
    X_train_all,
    y_train_all,
    test_size=0.2,
    random_state=42,
    stratify=y_train_all
)

# 只用训练集统计标准化
mean = X_train.mean(axis=(0, 2), keepdims=True)
std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6

X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std

X_train = torch.tensor(X_train).unsqueeze(1)
X_val = torch.tensor(X_val).unsqueeze(1)
X_test = torch.tensor(X_test).unsqueeze(1)

y_train = torch.tensor(y_train)
y_val = torch.tensor(y_val)
y_test = torch.tensor(y_test)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=64,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=64,
    shuffle=False
)

print("train batches:", len(train_loader))
print("val batches:", len(val_loader))
print("test batches:", len(test_loader))

test_12_ssvep (131, 8, 373) (131,) [41 39 51]
test_13_ssvep (88, 8, 373) (88,) [25 24 39]
test_14_ssvep (80, 8, 373) (80,) [36 20 24]
test_15_ssvep (85, 8, 373) (85,) [19 36 30]
test_11_ssvep (61, 8, 373) (61,) [18 25 18]
X_train_all: (384, 8, 373)
y_train_all: (384,)
X_test: (61, 8, 373)
y_test: (61,)
train batches: 20
val batches: 2
test batches: 1


# EEGNet

In [2]:
class EEGNet(nn.Module):
    def __init__(self, n_chans, n_classes, F1=8, D=2, F2=16, kern_len=64, dropout=0.5):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, kern_len), padding=(0, kern_len // 2), bias=False),
            nn.BatchNorm2d(F1),

            # depthwise spatial conv
            nn.Conv2d(F1, F1 * D, kernel_size=(n_chans, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout)
        )

        self.block2 = nn.Sequential(
            # separable temporal conv
            nn.Conv2d(F1 * D, F1 * D, kernel_size=(1, 16), padding=(0, 8),
                      groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(dropout)
        )

        self.classifier = None
        self.n_classes = n_classes

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = x.flatten(1)

        if self.classifier is None:
            self.classifier = nn.Linear(x.shape[1], self.n_classes).to(x.device)

        return self.classifier(x)

# 训练

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

n_chans = X_train.shape[2]
n_classes = len(freqs)

model = EEGNet(
    n_chans=n_chans,
    n_classes=n_classes,
    dropout=0.5
).to(device)

# 先创建 classifier
with torch.no_grad():
    dummy = torch.zeros(1, 1, n_chans, X_train.shape[-1]).to(device)
    _ = model(dummy)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)


def evaluate(model, loader):
    model.eval()
    all_pred = []
    all_true = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            pred = logits.argmax(dim=1).cpu()

            all_pred.append(pred)
            all_true.append(yb)

    y_pred = torch.cat(all_pred).numpy()
    y_true = torch.cat(all_true).numpy()

    acc = np.mean(y_pred == y_true)
    return acc, y_true, y_pred


best_val_acc = 0
best_state = None

for epoch in range(1, 201):
    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        pred = model(xb)
        loss = criterion(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * xb.size(0)

    train_loss /= len(train_loader.dataset)

    val_acc, _, _ = evaluate(model, val_loader)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if epoch % 10 == 0 or epoch == 1:
        print(
            f"Epoch {epoch:03d} "
            f"loss={train_loss:.4f} "
            f"val_acc={val_acc:.4f} "
            f"best_val={best_val_acc:.4f}"
        )

model.load_state_dict(best_state)
print("best val acc:", best_val_acc)

test_acc, y_true, y_pred = evaluate(model, test_loader)

print("final test acc:", test_acc)

for i, f in enumerate(freqs):
    mask = y_true == i
    acc_i = np.mean(y_pred[mask] == y_true[mask])
    print(f"{f} Hz test acc:", acc_i)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=["15", "12", "10"]))

pred_freqs = [freqs[i] for i in y_pred]
print(pred_freqs)

Epoch 001 loss=1.1137 val_acc=0.3506 best_val=0.3506
Epoch 010 loss=0.8926 val_acc=0.6623 best_val=0.6623
Epoch 020 loss=0.5346 val_acc=0.8052 best_val=0.8052
Epoch 030 loss=0.4413 val_acc=0.8571 best_val=0.8571
Epoch 040 loss=0.3142 val_acc=0.8961 best_val=0.9091
Epoch 050 loss=0.2530 val_acc=0.9221 best_val=0.9351
Epoch 060 loss=0.2474 val_acc=0.9351 best_val=0.9351
Epoch 070 loss=0.2067 val_acc=0.9351 best_val=0.9351
Epoch 080 loss=0.1813 val_acc=0.9221 best_val=0.9351
Epoch 090 loss=0.1664 val_acc=0.9351 best_val=0.9351
Epoch 100 loss=0.1690 val_acc=0.9221 best_val=0.9351
Epoch 110 loss=0.1498 val_acc=0.9221 best_val=0.9351
Epoch 120 loss=0.1446 val_acc=0.9351 best_val=0.9351
Epoch 130 loss=0.1252 val_acc=0.9221 best_val=0.9351
Epoch 140 loss=0.1486 val_acc=0.9221 best_val=0.9351
Epoch 150 loss=0.0960 val_acc=0.9221 best_val=0.9351
Epoch 160 loss=0.1207 val_acc=0.9221 best_val=0.9351
Epoch 170 loss=0.1192 val_acc=0.9221 best_val=0.9351
Epoch 180 loss=0.1082 val_acc=0.9221 best_val=